# R2 Checkpoint — Rule Dry-Run & Load-Test Plan

**Framework:** File Validation & Routing (`fvr`) • **Gate:** R2 — first end-to-end execution of Steps 3–4 (upload API + validation worker) against the **real volumes** (`mdd_platform.fvr.{staging,intake,quarantine}`) and the **real seed contracts** • **Prerequisites:** Steps 1–2 applied (registry + operational collections), Step 3 API deployed, Step 4 worker deployed

R2 has two parts that must BOTH pass:

| Part | Question it answers | Evidence |
| --- | --- | --- |
| **A. Rule dry-run** (cell 2) | do the 8 seed rules + 3 routing policies behave exactly as designed on real files? | test-file matrix — every case lands in the expected destination with the expected evidence rows |
| **B. Load test** (cells 3–4) | does the direct-to-volume flow hold up under concurrency, and where are its real limits? (Storage Addendum §4 predicted ✅ — now measure) | locust run vs thresholds; staging janitor leaves zero orphans |

**R2 passes when:** all 12 dry-run cases produce the expected `current_status` + destination + `validation_results`/`file_lifecycle` rows; every load threshold in cell 3 holds; and no Mongo or volume artifact is duplicated by any retry.

## 2. Part A — Rule dry-run test-file matrix

Prepare one session per row (all through the real API — never insert directly). Cases are cumulative over the 8 seed rules (`rule_*`) and 3 routing policies (`pol_*`).

| # | Input | Rule(s) exercised | Expected `current_status` | Expected destination | Evidence to assert |
| --- | --- | --- | --- | --- | --- |
| 1 | `orders_q3.csv` — correct 7-col header, 3 valid rows | FILE/HEADER/CONTENT happy path | `PASSED` → `ROUTED_INTAKE` | `intake/order/` renamed `ft_order_<file_id>_<ts>.csv` | `validation_results` run_seq 1 all PASS; transfer `MONGO_TO_INTAKE` COMPLETED; staging copy deleted |
| 2 | `customers_full.csv` — 10-col real contract, 1 row | customer happy path | `PASSED` → `ROUTED_INTAKE` | `intake/customer/` | as #1 |
| 3 | `payments_may.csv` — real 7-col payments contract | payments happy path | `PASSED` → `ROUTED_INTAKE` | `intake/payments/` | as #1 (incl. `card_last4` present — PII flagged, unmasked at rest) |
| 4 | header missing a column (`email` absent from customer file) | `SCHEMA_MATCH` (ERROR) | `FAILED` → `ROUTED_QUARANTINE` | `quarantine/` (via `pol_error_quarantine`) | FAIL row with `details.expected/actual` columns |
| 5 | columns present but reordered | `COLUMN_ORDER_MATCH` (ERROR) | `FAILED` → `ROUTED_QUARANTINE` | `quarantine/` | FAIL row with `details.order_mismatch` |
| 6 | valid header, **zero data rows** | `FILE_NOT_EMPTY_ROWS` (WARN — shadow) | `WARNED` → `ROUTED_INTAKE` (R1: WARN shadows at launch) | `intake/…` | WARN row present; file still routed; ops board shows the warning badge |
| 7 | `employee_roster.xlsx` | `FILE_TYPE_ALLOWED` (ERROR — extension) | `REJECTED` at manifest — **400, nothing staged** | none — no volume writes | error response carries the rule_id |
| 8 | 501 MB valid order file | `FILE_SIZE_MAX` (ERROR) | `REJECTED` at manifest — 413 | none | as #7 |
| 9 | 0-byte `orders_empty.csv` | `FILE_NOT_EMPTY` (ERROR) | `REJECTED` at manifest — 400 | none | as #7 |
| 10 | re-upload of case #1 byte-for-byte | `FILE_NOT_DUPLICATE` (ERROR, C1) | second complete → `duplicates:[file_id]`, session `COMPLETED_PARTIAL` | no second copy anywhere | original untouched; `sha256_checksum` unique index did the work |
| 11 | `random_export.csv` — header matches nothing | identification patterns | `UNIDENTIFIED` → `ROUTED_QUARANTINE` (via `pol_unidentified_quarantine`) | `quarantine/unidentified/` | `unidentified_reason` set; `identification_method: FILENAME_PATTERN` attempted |
| 12 | worker killed mid-validation (chaos case) | FRAMEWORK_ERROR retry | `FRAMEWORK_ERROR` → retried → normal outcome | eventual destination per rules | `retry_count` incremented; event `DEAD` **absent**; no poison spill before 3 attempts |

> **Sequencing note:** run cases 1–3 first (all PASS) so the happy path is proven before failure modes. Case 10 re-uses case 1's bytes. Case 12 needs `kill -9` on one worker pod while case 1 is in flight — proves C6/C7 (no lost file, no duplicate transfer).
> **Fixtures ready (2026-11-09):** all case files live in the repo at `fvr_test_files/` — `r2_manifest.json` carries each case's `file_type_code`, `declared_size`, and expected outcome, so the dry-run harness can drive the matrix straight from it. Case 8's stub is tiny on purpose: the 413 comes from the manifest's declared 501 MB, not the bytes.

After the matrix: run one **re-validate** on case 5's quarantined file (fix not applied) — expect `run_seq: 2` rows appended, originals immutable, status re-aggregated, audit trail carrying before/after.

## 3. Part B — Load-test plan (direct-to-volume flow)

**Objective:** validate the Storage Addendum §4 concurrency verdicts with numbers, and find the real Files-API rate ceiling before users do. **Environment:** dev MongoDB replica set + real `mdd_platform.fvr` volumes + one FastAPI pod (the concurrency question is storage/API contention, not pod count). **Data:** generated order/customer CSVs at 1 KB, 100 KB, 5 MB, 15 MB (75/20/4/1 mix — the small-file profile that drove the GridFS decision).

### Scenarios (locust, run in order)

| ID | Scenario | Shape | Duration | Proves |
| --- | --- | --- | --- |
| S1 | Baseline | 1 virtual user, sequential sessions | 10 min | per-stage latency floor, no-contention reference |
| S2 | Concurrent sessions | 10 VUs, each uploading a session | 10 min | no path collisions, per-session dirs hold, P95 stays sane |
| S3 | Pressure | 50 VUs mixed file sizes | 15 min | independent-object PUTs scale; expose 429s and their handling |
| S4 | Spike | 100 VUs for 5 min after a 10-VU warm-up | 5 min | retry/backoff absorbs a burst; no 5xx leakage to clients |
| S5 | Soak | 20 VUs sustained | 60 min | no connection/memory leak; janitor keeps staging dir count flat |

### Metrics & thresholds (all must hold for a pass)

| Metric | Where | Threshold |
| --- | --- | --- |
| Manifest P95 | API metrics | ≤ 500 ms |
| Stream-PUT P95 (15 MB file) | API metrics | ≤ 5 s end-to-end (≈3 MB/s effective) |
| Finalize P95 (per file, txn incl.) | API metrics | ≤ 2 s |
| End-to-end (upload→`ROUTED_INTAKE`) P95 | worker metrics | ≤ 30 s at S3 |
| Sustained throughput | locust | ≥ 300 files/hr at S3 (baseline target; document actual) |
| Error rate | locust | < 1% excluding intentional 4xx (dup/oversize test traffic) |
| Files API 429/5xx | `files_api_429_total` | allowed, but **zero** client-visible 502/503 after retries |
| Mongo ops | `serverStatus().opcounters` | no runaway; write latency P95 < 20 ms |
| Staging orphans | `staging_dir_count` after janitor | 0 abandoned session dirs 5 min post-S5 |
| Duplication invariants | spot SQL/mongosh | 0 duplicate `sha256_checksum`; 0 double `COMPLETED` transfers; 1 `FILE_ARRIVED` per file |

**Abort criteria:** client-visible 5xx > 2% for 2 consecutive minutes, or any evidence of a lost file (registry row without staged bytes or vice versa, excluding known dup cases) — stop, diagnose with the correlation_id trail, fix, re-run from S1.

### Locust sketch

```python
from locust import HttpUser, task, between
import uuid

class Uploader(HttpUser):
    wait_time = between(0.5, 2)

    @task
    def upload_session(self):
        sid_file = self.client.post("/upload-sessions", json={
            "file_type_code": "order",
            "files": [{"original_name": f"orders_{uuid.uuid4().hex[:6]}.csv",
                       "declared_size": 102400}]}).json()
        f = sid_file["files"][0]
        self.client.put(f["upload_url"],
                        data=csv_bytes(100_000),                    # 100 KB profile row
                        headers={"Content-Type": "application/octet-stream"})
        self.client.post(f"/upload-sessions/{sid_file['session_id']}/complete")
```

## 4. Execution checklist & sign-off

| # | Step | Owner | Done |
| --- | --- | --- | --- |
| 1 | Steps 1–2 verified (all collections `validationAction: error`, seeds 1/5/5/8/3/7) | — | ☐ |
| 2 | Step 3 API + Step 4 worker deployed to dev; SP has `WRITE FILES` on staging (API) and on intake/quarantine (worker) | — | ☐ |
| 3 | Test files for matrix cases 1–11 generated + case-12 kill script ready | — | ☐ |
| 4 | Part A matrix executed; 12/12 expected outcomes + re-validate follow-up verified | — | ☐ |
| 5 | S1–S5 executed in order; metrics captured; every threshold met (or deviation documented with cause + fix) | — | ☐ |
| 6 | Janitor run post-S5: staging clean; no orphan dirs | — | ☐ |
| 7 | Results reviewed; issues logged below; **R2 verdict recorded** | — | ☐ |

**Issues log:** | id | case/scenario | symptom | root cause | fix | follow-up |
| --- | --- | --- | --- | --- | --- |
| | | | | | |

> **Post-R2 decision point (Storage Addendum open item #4):** if sustained throughput need exceeds ~3× the measured S3 ceiling, promote validation/move into a Databricks-side job (same collections, same contracts — no model change). Decide with the R2 numbers, not speculation.

## 5. Pre-R2 execution log (2026-11-09) — engine-logic dry-run on real volumes

Before the deployed stack exists, the **engine logic of the matrix** was executed from Databricks compute against the real fixtures and the **live `mdd_platform.fvr` volumes** (replicating the Step-3/4 stages: pre-staging checks → stage → sha256 dedup → identify → header/content rules → aggregate → route → move + hash-verify → staging cleanup):

| Case | Simulated outcome | Verdict |
| --- | --- | --- |
| 1–3 | `PASSED → ROUTED_INTAKE` (order/customer/payments happy paths; destination PUT hash-verified) | ✅ |
| 4 | `FAILED (SCHEMA_MATCH) → ROUTED_QUARANTINE` | ✅ |
| 5 | `FAILED (COLUMN_ORDER_MATCH) → ROUTED_QUARANTINE` | ✅ |
| 6 | `WARNED (FILE_NOT_EMPTY_ROWS) → ROUTED_INTAKE` (WARN shadow) | ✅ |
| 7/8/9 | rejected at manifest (extension 400 / declared-501MB 413 / empty 400) — **no bytes written** | ✅ |
| 10 | `DUPLICATE (sha256 = original file) → COMPLETED_PARTIAL`, no second copy | ✅ |
| 11 | `UNIDENTIFIED → ROUTED_QUARANTINE` | ✅ |
| 12 | worker-kill chaos — **BLOCKED: needs the deployed worker** | ⏸ |

**Result: 11/11 executable cases PASS, 0 FAIL, 1 blocked.** Volumes left pristine (all simulation artifacts purged and verified).

**Indicative concurrency probe (Part B pre-signal):** 40 × 100 KB files, 20 concurrent writers on the real staging volume — 40/40 hash-verified, ~13 files/s aggregate, ~75 ms/file for the full write+read+verify+delete cycle, no contention. Measures the storage layer via compute FUSE only — the deployed service adds the Files API HTTPS hop, so **locust S1–S5 remain the authoritative gate**.

**What still requires the deployed stack (see the Deployment Runbook notebook):** the full-API matrix (409/413/422 error paths, ack-after-durable-write), the Mongo transaction/event trail, case 12 chaos, re-validate (`run_seq: 2`), and the entire locust plan.